## Itent-to-treat and Treatment effect in high dimensional setting

#### This part try to estimate the itent-to-treat as well as the treatment effect using high dimensioanl machine learning methods

 we start by creating relevant variables which will serve as covariates for our model

In [ ]:
# imports
import pandas as pd
import numpy as np

# load data
df = pd.read_stata("dataPrivatePublic.dta")
data = df.copy()

# region variables
data["IdF"] = (data["nregion"] == "116").astype(int)
data["North"] = (data["nregion"] == "311").astype(int)
data["Otherregion"] = 1 - data["IdF"] - data["North"]

# reason for job loss
data["EconLayoff"] = (data["motins"] == "1").astype(int)
data["PersLayoff"] = (data["motins"] == "2").astype(int)
data["EndCDD"] = (data["motins"] == "4").astype(int)
data["EndInterim"] = (data["motins"] == "5").astype(int)

data["Otherend"] = 1 - data[
    ["EconLayoff","PersLayoff","EndCDD","EndInterim"]
].sum(axis=1)

# experience
data["exper0"] = (data["exper"] == "00").astype(int)

data["exper1_5"] = data["exper"].isin(
    ["01","02","03","04","05"]
).astype(int)

data["experM5"] = 1 - data["exper0"] - data["exper1_5"]

# statistical risk group
data["rsqstat2"] = (data["rsqstat"] == "RS2").astype(int)
data["rsqstat3"] = (data["rsqstat"] == "RS3").astype(int)

data["Orsqstat"] = 1 - data["rsqstat2"] - data["rsqstat3"]

# job search type
data["tempcomp"] = (data["temps"] == "1").astype(int)
data["Otemp"] = 1 - data["tempcomp"]

# sensitive area
data["dezus"] = (data["zus"] == "ZU").astype(int)

# wage expectations
for s in ["A","B","C","D","E"]:
    data[f"salaire{s}"] = (data["salaire"] == s).astype(int)

data["salaireG"] = data["salaire"].isin(["G",""]).astype(int)

# job type
data["ce1"] = (data["cemploi"] == "CE1").astype(int)
data["ce2"] = (data["cemploi"] == "CE2").astype(int)
data["cemiss"] = (data["cemploi"] == "").astype(int)

# first time unemployed
data["primo"] = (data["ndem"] == 1).astype(int)

# socioprofessional category
data["Cadre"] = (data["CS"] == 3).astype(int)
data["Techn"] = (data["CS"] == 4).astype(int)

data["EmployQ"] = (data["CS"] == 51).astype(int)
data["EmployNQ"] = (data["CS"] == 56).astype(int)

data["OuvrQ"] = (data["CS"] == 61).astype(int)
data["OuvrNQ"] = data["CS"].isin([66,99]).astype(int)

# nationality groups
data["African"] = (
    (data["nation"] >= "31") & (data["nation"] <= "49")
).astype(int)

data["EasternEurope"] = (
    ((data["nation"] >= "90") & (data["nation"] <= "98")) |
    (data["nation"].isin(["24","25","27"]))
).astype(int)

data["SouthEuropTurkey"] = data["nation"].isin(
    ["02","03","14","19","21","22","24","27","26"]
).astype(int)

# children
data["nochild"] = (data["nenf"] == 0).astype(int)
data["onechild"] = (data["nenf"] == 1).astype(int)
data["twoormorechild"] = (data["nenf"] > 1).astype(int)

# sex
data["woman"] = (data["sexe"] == "2").astype(int)

# randomization cohort
data["Q1"] = data["mois_saisie_occ"].between(1,3).astype(int)
data["Q2"] = data["mois_saisie_occ"].between(4,6).astype(int)
data["Q3"] = data["mois_saisie_occ"].between(7,9).astype(int)
data["Q4"] = data["mois_saisie_occ"].between(10,12).astype(int)

# cost variables
data["cost_opp"] = (
    data["acceptationOPP"] *
    (0.3 + data["EMPLOI_AR110_6MOIS"]*0.35 + 0.35*data["SUCCES_OPP_6MOIS"]) *
    3000
)

data["cost_cve"] = data["acceptationCVE"] * 657

data["cost_cla"] = (
    (1 - data["acceptationOPP"] - data["acceptationCVE"]) * 120
)

data["cost"] = (
    data["cost_opp"] +
    data["cost_cve"] +
    data["cost_cla"]
)

# save prepared dataset
data.to_parquet("prepared_private_public.parquet")

print("Dataset prepared successfully")

In [ ]:
# list of covariates

covariates = [

    # region
    "IdF",
    "North",
    "Otherregion",

    # reason for job loss
    "EconLayoff",
    "PersLayoff",
    "EndCDD",
    "EndInterim",
    "Otherend",

    # experience
    "exper0",
    "exper1_5",
    "experM5",

    # statistical risk score
    "rsqstat2",
    "rsqstat3",
    "Orsqstat",

    # job search type
    "tempcomp",
    "Otemp",

    # sensitive area
    "dezus",

    # wage expectations
    "salaireA",
    "salaireB",
    "salaireC",
    "salaireD",
    "salaireE",
    "salaireG",

    # job type searched
    "ce1",
    "ce2",
    "cemiss",

    # first time unemployed
    "primo",

    # socioprofessional category
    "Cadre",
    "Techn",
    "EmployQ",
    "EmployNQ",
    "OuvrQ",
    "OuvrNQ",

    # nationality groups
    "African",
    "EasternEurope",
    "SouthEuropTurkey",

    # family
    "nochild",
    "onechild",
    "twoormorechild",

    # gender
    "woman",

    # cohort of randomization
    "Q1",
    "Q2",
    "Q3",
    "Q4",
    
    # age class
    "agegr2635",
    "agegr3645",
    "agegr4655",
    "agegr56"
]